In [ ]:
"""
Post-Training Quantization (PTQ) v2 — ResNet-50 / CIFAR-10
===========================================================
Takes a trained FP32 model and applies quantization AFTER training.
No gradient updates — only calibration of scale S and zero-point Z.

Mathematical foundation:
  Q(x)  = round(x / S) + Z          [quantization — maps float → integer]
  x̂     = S · (Q(x) − Z)            [dequantization — reconstruction at inference]
  ε     = x − x̂,  |ε| ≤ S/2        [quantization error — bounded by step size]
  S     = (x_max − x_min) / (2^b−1) [scale — divides observed range over all levels]
  Z     = round(−x_min / S)         [zero-point — maps real 0 to an integer]

  INT8: 2^8 = 256 levels, S is small → fine-grained, error small
  INT4: 2^4 = 16 levels,  S is large → coarse, error accumulates per layer

Three back-ends benchmarked:
  1. Dynamic      — Linear weights → qint8; activations FP32 (no calibration)
  2. Static (FX)  — Conv2d + Linear weights + activations → INT8
                    Sweeps 4 observer types: minmax, moving_avg, histogram, percentile
  3. FX-Graph     — torch.ao graph-mode with x86/fbgemm default qconfigs

Fixes from v1 review:
  FIX-1  inference_ms       — Now measures single-sample (batch_size=1) latency,
                              matching the pruning experiment convention. v1 used
                              batch_size=128, making cross-experiment comparison invalid.
  FIX-2  FLOPs via thop     — GFLOPs now computed for every method. PTQ does not
                              change the compute graph (same as mask-based pruning),
                              so FLOPs equal the baseline. Stated explicitly + measured.
  FIX-3  inline baseline    — FP32 baseline accuracy re-evaluated in this script on
                              the same test loader, rather than loaded from a JSON
                              file that may have been computed under different conditions.
  FIX-4  deduplication      — Model-saving block now reuses the already-quantized
                              models from the main loop instead of re-running all
                              quantization passes from scratch.

Output: __1__PTQ_v2.json
        __1__PTQ_quantized_models/  (one .pt per method)
"""

In [1]:
import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.quantization as tq
from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from thop import profile

warnings.filterwarnings("ignore")

# ── Config ────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH = "../__2__baseline_resnet50_cifar10.pth"
OUTPUT_JSON         = "__1__PTQ_v2.json"
MODEL_SAVE_DIR      = "__1__PTQ_quantized_models"

DEVICE         = torch.device("cpu")   # PyTorch static PTQ is CPU-only
BATCH_SIZE     = 128
NUM_WORKERS    = 2
NUM_CLASSES    = 10
CALIB_SIZE     = 1024    # images used to calibrate static observers
CALIB_BATCHES  = 8       # max forward passes during calibration
INFERENCE_RUNS = 200     # repetitions for latency measurement (single-sample)
WARMUP_RUNS    = 20      # warmup passes before timing

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── Observer configs for static PTQ sweep ─────────────────────────────────────
from torch.ao.quantization import QConfig

OBSERVER_QCONFIGS = {
    "minmax": QConfig(
        activation=tq.MinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "moving_avg": QConfig(
        activation=tq.MovingAverageMinMaxObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.MovingAveragePerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "histogram": QConfig(
        activation=tq.HistogramObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
    "percentile": QConfig(
        activation=tq.HistogramObserver.with_args(
            dtype=torch.quint8, qscheme=torch.per_tensor_affine,
            reduce_range=True),
        weight=tq.PerChannelMinMaxObserver.with_args(
            dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
    ),
}

# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str) -> nn.Module:
    model = build_model().cpu()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    return model

# ── Data ──────────────────────────────────────────────────────────────────────
def get_test_loader() -> DataLoader:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False)

def get_calib_loader() -> DataLoader:
    """
    Small representative subset from training split.
    Observers watch this data to compute x_min and x_max,
    from which S = (x_max - x_min) / 255 is derived.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds  = torchvision.datasets.CIFAR10(root="../data", train=True,
                                        download=True, transform=transform)
    sub = Subset(ds, list(range(CALIB_SIZE)))
    return DataLoader(sub, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=False)

# ── Utilities ─────────────────────────────────────────────────────────────────
def disk_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        # torch.save on the full model (not state_dict) for quantized models —
        # state_dict() alone does not preserve INT8 tensor packing.
        torch.save(model, tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def fp32_ram_mb(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) * 4 / 1024 ** 2

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        preds.extend(model(inputs).argmax(1).numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

# FIX-1: Single-sample latency, matching pruning experiment convention.
# v1 used batch_size=128, which measures throughput not latency, and cannot
# be compared to the pruning script's single-image inference_ms.
# No CUDA synchronization needed here — PTQ is CPU-only.
def measure_latency_ms(model: nn.Module,
                       n: int = INFERENCE_RUNS,
                       warmup: int = WARMUP_RUNS) -> float:
    """
    Single-sample (batch_size=1) inference latency in milliseconds.
    Matches the measurement convention in the pruning experiments (v5).
    PTQ is CPU-only, so no torch.cuda.synchronize() needed.
    """
    model = model.cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)   # single sample, not a batch
    with torch.no_grad():
        for _ in range(warmup):
            model(dummy)
        t0 = time.perf_counter()
        for _ in range(n):
            model(dummy)
    return float((time.perf_counter() - t0) / n * 1000)

# FIX-2: GFLOPs computation via thop, matching pruning experiments.
# PTQ does not change the compute graph — FLOPs equal baseline.
# We measure anyway so the number appears in the JSON for apples-to-apples
# comparison and so the FLOPS_NOTE_QUANT disclosure is verified by data.
def compute_flops(model: nn.Module) -> float:
    """GFLOPs for a single 32×32 image forward pass (matches pruning script)."""
    model.eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = profile(model, inputs=(dummy,), verbose=False)
        return float(macs * 2 / 1e9)
    except Exception:
        # thop may fail on quantized ops in some PyTorch versions;
        # fall back to FP32 baseline estimate and note it.
        return -1.0   # sentinel; will be replaced in build_row

FLOPS_NOTE_QUANT = (
    "PTQ (all backends) does not change the computation graph — "
    "MAC count is identical to the FP32 baseline. "
    "INT8 arithmetic yields ~2-4× throughput on CPUs with AVX-512 VNNI "
    "because 4 INT8 MACs fit in one VNNI instruction vs 1 FP32 FMA. "
    "GFLOPs reported here are dense FP32-equivalent MACs × 2; "
    "effective INT8 FLOPs from hardware counters would be ~4× lower."
)

@torch.no_grad()
def calibrate(model: nn.Module, loader: DataLoader,
              max_batches: int = CALIB_BATCHES) -> None:
    """
    Calibration pass: runs representative data so observers can collect
    activation statistics (x_min, x_max) and compute S and Z.
    S = (x_max - x_min) / (2^b - 1)
    Z = round(-x_min / S)
    """
    model.eval()
    for i, (inputs, _) in enumerate(loader):
        model(inputs)
        if i + 1 >= max_batches:
            break

def build_row(backend, observer, description, metrics, baseline_acc,
              base_disk, base_flops, disk_mb, latency_ms, flops_g,
              layers_quantized, **extra) -> dict:
    # If thop failed on quantized model, report baseline FLOPs with a note
    reported_flops = flops_g if flops_g > 0 else base_flops
    flops_source   = "measured" if flops_g > 0 else "baseline_estimate (thop unsupported for this quant op)"
    return {
        "backend"             : backend,
        "observer"            : observer,
        "description"         : description,
        "accuracy"            : round(metrics["accuracy"],  6),
        "precision"           : round(metrics["precision"], 6),
        "recall"              : round(metrics["recall"],    6),
        "f1"                  : round(metrics["f1"],        6),
        "accuracy_drop"       : round(baseline_acc - metrics["accuracy"], 6),
        "size_disk_mb"        : round(disk_mb, 4),
        "disk_saved_mb"       : round(base_disk - disk_mb, 4),
        "compression_ratio"   : round(base_disk / disk_mb, 4) if disk_mb else None,
        "flops_G"             : round(reported_flops, 4),
        "flops_source"        : flops_source,
        "flops_note"          : FLOPS_NOTE_QUANT,
        "inference_ms"        : round(latency_ms, 4),
        "inference_ms_note"   : (
            "Single-sample CPU latency (batch_size=1), matching pruning v5 convention. "
            "200 timed passes after 20 warmup passes. CPU-only; no CUDA sync needed."
        ),
        "layers_quantized"    : layers_quantized,
        **extra,
    }

# ══════════════════════════════════════════════════════════════════════════════
# 1. Dynamic PTQ
# ──────────────────────────────────────────────────────────────────────────────
# What:    Weights of nn.Linear layers → qint8
# S/Z:     Computed on-the-fly per inference batch (not from calibration data)
# Formula: Q(x) = round(x/S) + Z  applied to Linear weight tensors only
# When:    No calibration data available; quick to apply; lower accuracy gain
#          than static because Conv2d stays FP32
# ══════════════════════════════════════════════════════════════════════════════
def run_dynamic_ptq(fp32_model, test_loader, baseline_acc,
                    base_disk, base_flops) -> tuple:
    print("\n  [1/3] Dynamic PTQ")
    print("        Quantizes  : Linear weights → qint8")
    print("        Activations: FP32 (scale computed per batch, on-the-fly)")
    print("        Calibration: none required")

    q_model = torch.quantization.quantize_dynamic(
        copy.deepcopy(fp32_model).cpu(), {nn.Linear}, dtype=torch.qint8
    )
    q_model.eval()

    metrics    = evaluate(q_model, test_loader)
    latency_ms = measure_latency_ms(q_model)
    disk_mb    = disk_size_mb(q_model)
    flops_g    = compute_flops(q_model)

    row = build_row(
        backend          = "dynamic",
        observer         = "per_channel_affine (runtime)",
        description      = (
            "Dynamic PTQ: only Linear weights quantized to qint8. "
            "S and Z computed on-the-fly per batch — no calibration data. "
            "Conv2d stays FP32. Fast to apply; no memory of x_min/x_max needed."
        ),
        metrics          = metrics,
        baseline_acc     = baseline_acc,
        base_disk        = base_disk,
        base_flops       = base_flops,
        disk_mb          = disk_mb,
        latency_ms       = latency_ms,
        flops_g          = flops_g,
        layers_quantized = "Linear only",
        calibration_samples = 0,
        quantization_formulas = {
            "Q(x)"       : "round(x / S) + Z",
            "x̂"          : "S · (Q(x) − Z)",
            "ε"          : "|x − x̂| ≤ S/2",
            "S"          : "computed per-batch at runtime from weight range",
            "dtype_w"    : "qint8  (8-bit signed integer)",
            "dtype_acts" : "float32 (not quantized)",
        },
    )
    print(f"        → Acc: {metrics['accuracy']:.4f}  "
          f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
          f"Disk: {disk_mb:.2f} MB  Lat: {latency_ms:.2f} ms  "
          f"GFLOPs: {flops_g:.3f}")
    return row, q_model

# ══════════════════════════════════════════════════════════════════════════════
# 2. Static PTQ — FX-graph mode, sweeping 4 observer strategies
# ──────────────────────────────────────────────────────────────────────────────
# What:    Conv2d + Linear weights AND activations → INT8
# S/Z:     Determined ONCE from calibration data, then fixed for all inference
#          S = (x_max − x_min) / (2^b − 1)   [b=8 → 255 quantization levels]
#          Z = round(−x_min / S)
# Graph:   FX tracing fuses Conv-BN-ReLU and residual adds automatically
# Sweep:   4 observer types to compare x_min/x_max estimation strategies
# ══════════════════════════════════════════════════════════════════════════════
def run_static_ptq(fp32_model, test_loader, calib_loader,
                   baseline_acc, base_disk, base_flops) -> tuple:
    print("\n  [2/3] Static PTQ — FX-graph mode  (sweeping 4 observers)")
    print("        Quantizes : Conv2d + Linear + residual adds (weights + activations → INT8)")
    print(f"        Scale S  = (x_max − x_min) / (2^8 − 1)  from {CALIB_SIZE} calib images")
    rows, models_out = [], {}
    example = torch.randn(1, 3, 32, 32)

    for obs_name, qconfig in OBSERVER_QCONFIGS.items():
        print(f"        Observer: {obs_name:12s}", end="", flush=True)
        try:
            model    = copy.deepcopy(fp32_model).cpu().eval()
            prepared = prepare_fx(model, {"": qconfig}, example)
            calibrate(prepared, calib_loader)
            q_model  = convert_fx(prepared)
            q_model.eval()

            metrics    = evaluate(q_model, test_loader)
            latency_ms = measure_latency_ms(q_model)
            disk_mb    = disk_size_mb(q_model)
            flops_g    = compute_flops(q_model)

            row = build_row(
                backend          = "static_fx",
                observer         = obs_name,
                description      = (
                    f"Static PTQ (FX-graph, {obs_name}): graph-traced fusion of "
                    f"Conv-BN-ReLU and residual adds. "
                    f"S and Z calibrated from {CALIB_SIZE} training images. "
                    "Weights and activations quantized to INT8."
                ),
                metrics          = metrics,
                baseline_acc     = baseline_acc,
                base_disk        = base_disk,
                base_flops       = base_flops,
                disk_mb          = disk_mb,
                latency_ms       = latency_ms,
                flops_g          = flops_g,
                layers_quantized = "Conv2d + Linear + residual adds (weights + activations)",
                calibration_samples = CALIB_SIZE,
                quantization_formulas = {
                    "S"        : "S = (x_max − x_min) / (2^8 − 1)  [255 levels for INT8]",
                    "Z"        : "Z = round(−x_min / S)",
                    "Q(x)"     : "Q(x) = round(x / S) + Z",
                    "x̂"        : "x̂ = S · (Q(x) − Z)",
                    "max_err"  : "|ε| ≤ S/2  →  smaller S = finer grid = less error",
                    "dtype_w"  : "qint8",
                    "dtype_act": "quint8",
                },
            )
            models_out[f"static_{obs_name}"] = q_model
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  Lat: {latency_ms:.2f} ms  "
                  f"GFLOPs: {flops_g:.3f}")
        except Exception as e:
            print(f" → FAILED: {e}")
            row = {
                "backend"       : "static_fx",
                "observer"      : obs_name,
                "description"   : f"FAILED: {e}",
                "accuracy"      : None,
                "disk_saved_mb" : None,
            }
        rows.append(row)
    return rows, models_out

# ══════════════════════════════════════════════════════════════════════════════
# 3. FX-Graph Static PTQ — torch.ao default qconfigs (x86 / fbgemm)
# ──────────────────────────────────────────────────────────────────────────────
# Same S/Z math as static sweep above, but uses torch.ao's recommended
# default qconfigs tuned for specific CPU backends:
#   x86    → targets AVX-512 VNNI (INT8 throughput ~4× FP32)
#   fbgemm → x86 fallback, also used by Facebook's inference stack
# ══════════════════════════════════════════════════════════════════════════════
def run_fx_ptq(fp32_model, test_loader, calib_loader,
               baseline_acc, base_disk, base_flops) -> tuple:
    print("\n  [3/3] FX-Graph Static PTQ (torch.ao default configs) — x86 + fbgemm")
    print("        Uses backend-tuned qconfigs; broader op coverage than observer sweep")
    rows, models_out = [], {}
    example = torch.randn(1, 3, 32, 32)

    for backend in ["x86", "fbgemm"]:
        print(f"        Backend: {backend:10s}", end="", flush=True)
        try:
            model    = copy.deepcopy(fp32_model).cpu().eval()
            prepared = prepare_fx(model, {"": get_default_qconfig(backend)}, example)
            calibrate(prepared, calib_loader)
            q_model  = convert_fx(prepared)
            q_model.eval()

            metrics    = evaluate(q_model, test_loader)
            latency_ms = measure_latency_ms(q_model)
            disk_mb    = disk_size_mb(q_model)
            flops_g    = compute_flops(q_model)

            row = build_row(
                backend          = f"fx_static_{backend}",
                observer         = f"default_qconfig({backend})",
                description      = (
                    f"FX-graph static PTQ (backend={backend}): "
                    "torch.ao traces the computation graph, automatically fuses "
                    "Conv-BN-ReLU and quantizes all eligible ops including residual adds. "
                    "Same S/Z formulas as eager static; backend-tuned qconfig."
                ),
                metrics          = metrics,
                baseline_acc     = baseline_acc,
                base_disk        = base_disk,
                base_flops       = base_flops,
                disk_mb          = disk_mb,
                latency_ms       = latency_ms,
                flops_g          = flops_g,
                layers_quantized = "Conv2d + Linear + residual adds (auto-fused)",
                calibration_samples = CALIB_SIZE,
                quantization_formulas = {
                    "backend"    : backend,
                    "fusion"     : "automatic via FX graph tracing",
                    "dtype"      : "qint8 (weights), quint8 (activations)",
                    "throughput" : "INT8 VNNI ~4× FP32 on AVX-512 CPUs",
                },
            )
            models_out[f"fx_{backend}"] = q_model
            print(f" → Acc: {metrics['accuracy']:.4f}  "
                  f"Drop: {baseline_acc - metrics['accuracy']:+.4f}  "
                  f"Disk: {disk_mb:.2f} MB  Lat: {latency_ms:.2f} ms  "
                  f"GFLOPs: {flops_g:.3f}")
        except Exception as e:
            print(f" → FAILED: {e}")
            row = {
                "backend"      : f"fx_static_{backend}",
                "observer"     : f"default_qconfig({backend})",
                "description"  : f"FAILED: {str(e)}",
                "accuracy"     : None,
                "disk_saved_mb": None,
            }
        rows.append(row)
    return rows, models_out

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  PTQ v2 — ResNet-50 / CIFAR-10")
    print("  Math: Q(x)=round(x/S)+Z | S=(x_max-x_min)/(2^b-1) | |ε|≤S/2")
    print(f"  Device: CPU  |  Calibration: {CALIB_SIZE} images  |  Latency: single-sample")
    print(f"{'='*65}")

    fp32_model   = load_baseline(BASELINE_MODEL_PATH)
    test_loader  = get_test_loader()
    calib_loader = get_calib_loader()

    # FIX-3: Re-evaluate baseline inline on the same test loader.
    # Do NOT load accuracy from a pre-saved JSON — different transforms,
    # batch sizes, or data splits would make accuracy_drop misleading.
    print("\n  Evaluating FP32 baseline ...")
    baseline_metrics = evaluate(fp32_model, test_loader)
    baseline_acc     = baseline_metrics["accuracy"]
    base_disk        = disk_size_mb(fp32_model)
    base_ram         = fp32_ram_mb(fp32_model)
    base_latency     = measure_latency_ms(fp32_model)
    base_flops       = compute_flops(fp32_model)
    print(f"  Baseline: Acc={baseline_acc:.4f}  Disk={base_disk:.2f} MB  "
          f"RAM={base_ram:.2f} MB  Lat={base_latency:.2f} ms  GFLOPs={base_flops:.3f}")

    # Run all three PTQ backends
    # FIX-4: each runner returns (rows, quantized_models_dict) so we reuse
    # the already-quantized models for saving — no second quantization pass.
    all_quantized_models = {}

    dyn_row, dyn_model = run_dynamic_ptq(
        fp32_model, test_loader, baseline_acc, base_disk, base_flops)
    all_quantized_models["dynamic"] = dyn_model

    static_rows, static_models = run_static_ptq(
        fp32_model, test_loader, calib_loader, baseline_acc, base_disk, base_flops)
    all_quantized_models.update(static_models)

    fx_rows, fx_models = run_fx_ptq(
        fp32_model, test_loader, calib_loader, baseline_acc, base_disk, base_flops)
    all_quantized_models.update(fx_models)

    results = [dyn_row] + static_rows + fx_rows

    # ── Best config ───────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("\n  ✗ All backends failed. Check PyTorch version compatibility.")
        return

    # Best = lowest accuracy drop; break ties by smallest disk size
    best = min(valid, key=lambda r: (r["accuracy_drop"], r["size_disk_mb"]))

    # ── Save JSON ─────────────────────────────────────────────────────────────
    report = {
        "method"      : "post_training_quantization_v2",
        "description" : (
            "PTQ v2: quantization applied after training with no gradient updates. "
            "Scale S and zero-point Z derived from calibration data (static) or "
            "computed per-batch at runtime (dynamic). "
            "Three back-ends: dynamic (no calib), static FX sweep (4 observers), "
            "FX-graph with backend-tuned default qconfigs. "
            "Latency measured single-sample to match pruning v5. "
            "FLOPs computed via thop (graph unchanged — values equal baseline)."
        ),
        "ptq_math": {
            "quantize"    : "Q(x) = round(x / S) + Z",
            "dequantize"  : "x̂ = S · (Q(x) − Z)",
            "error"       : "ε = x − x̂,  |ε| ≤ S/2",
            "scale"       : "S = (x_max − x_min) / (2^b − 1)",
            "zero_point"  : "Z = round(−x_min / S)",
            "int8_levels" : "2^8 − 1 = 255",
            "int4_levels" : "2^4 − 1 = 15",
            "note"        : (
                "INT8 works well with PTQ; INT4 degrades because 15 levels "
                "gives large S → large |ε|, and errors accumulate through layers "
                "with no mechanism to correct them (that is QAT's job)."
            ),
        },
        "calibration_samples" : CALIB_SIZE,
        "calibration_batches" : CALIB_BATCHES,
        "target_dtype"        : "qint8 (weights) / quint8 (activations)",
        "baseline": {
            **baseline_metrics,
            "size_disk_mb"   : round(base_disk, 4),
            "ram_fp32_mb"    : round(base_ram, 4),
            "inference_ms"   : round(base_latency, 4),
            "flops_G"        : round(base_flops, 4),
            "eval_source"    : "re-evaluated inline (FIX-3) — not loaded from external JSON",
        },
        "device"     : "cpu",
        "best_config": {
            "backend"          : best["backend"],
            "observer"         : best["observer"],
            "accuracy"         : best["accuracy"],
            "accuracy_drop"    : best["accuracy_drop"],
            "size_disk_mb"     : best["size_disk_mb"],
            "compression_ratio": best.get("compression_ratio"),
            "inference_ms"     : best.get("inference_ms"),
            "flops_G"          : best.get("flops_G"),
        },
        "results": results,
        "fixes_from_v1": {
            "FIX_1_single_sample_latency": (
                "inference_ms now uses batch_size=1 (single-sample), matching pruning v5. "
                "v1 used batch_size=128 — that's batch throughput, not latency."
            ),
            "FIX_2_flops": (
                "GFLOPs now computed per method via thop. "
                "PTQ does not change the graph — values match baseline. "
                "Explicitly stated in flops_note and flops_source per row."
            ),
            "FIX_3_inline_baseline": (
                "Baseline accuracy re-evaluated on the same test loader in this script. "
                "v1 loaded accuracy from an external JSON, risking stale/mismatched values."
            ),
            "FIX_4_deduplication": (
                "Quantized models returned from each runner and reused for saving. "
                "v1 re-ran all quantization passes from scratch in the saving block."
            ),
        },
        "notes": {
            "dynamic"       : "No calibration; Linear only; fastest to apply; Conv stays FP32",
            "static_fx"     : "Calibration required; best INT8 accuracy; Conv-BN-ReLU auto-fused by FX",
            "fx_default"    : "Backend-tuned qconfig; broadest op coverage; model must be torch.fx-traceable",
            "gpu_note"      : "PyTorch static PTQ is CPU-only; GPU INT8 → TensorRT / ONNX Runtime",
            "speedup_note"  : "INT8 ops ~2-4× faster than FP32 on CPUs with AVX-512 VNNI support",
            "vs_qat"        : (
                "PTQ requires no retraining and is fast; QAT closes the accuracy gap at INT4 "
                "by inserting fake-quant nodes during training and using the STE trick."
            ),
            "vs_pruning"    : (
                "PTQ and mask-based pruning both leave the compute graph unchanged. "
                "PTQ reduces arithmetic precision (FP32→INT8); pruning introduces sparsity. "
                "PTQ achieves ~4× disk compression with near-zero accuracy drop. "
                "Pruning at 50% sparsity achieves ~2× compression only with sparse kernels. "
                "Comparison: flops_G should be equal across both; latency depends on hardware."
            ),
            "disk_size_note": (
                "disk_size_mb uses torch.save(model) (full model, not state_dict) "
                "to ensure INT8 tensor packing is preserved in the serialized file."
            ),
            "flops_note"    : FLOPS_NOTE_QUANT,
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    # ── FIX-4: Save quantized models reusing already-built objects ────────────
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
    print(f"\n  Saving quantized models to {MODEL_SAVE_DIR}/ ...")
    for name, model in all_quantized_models.items():
        path = os.path.join(MODEL_SAVE_DIR, f"resnet50_{name}.pt")
        try:
            torch.save(model, path)
            size_mb = os.path.getsize(path) / 1024 ** 2
            print(f"    ✓ {os.path.basename(path):<45s} {size_mb:.2f} MB")
        except Exception as e:
            print(f"    ✗ {name}: {e}")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  ALL DONE — {OUTPUT_JSON}")
    print(f"{'='*65}")
    hdr = (f"  {'Backend':<22} {'Observer':<25} {'Acc':>7} {'Drop':>7} "
           f"{'MB':>7} {'GFLOPs':>8} {'ms':>7}")
    print("\n" + hdr)
    print("  " + "-" * (len(hdr)-2))

    # Print baseline row for reference
    print(f"  {'FP32 baseline':<22} {'—':<25} "
          f"{baseline_acc:>7.4f} {'0.0000':>7} "
          f"{base_disk:>7.2f} {base_flops:>8.3f} {base_latency:>7.2f}")

    for r in valid:
        flops_str = f"{r['flops_G']:>8.3f}" if r.get('flops_G', -1) > 0 else f"{'—':>8}"
        print(f"  {r['backend']:<22} {r['observer']:<25} "
              f"{r['accuracy']:>7.4f} {r['accuracy_drop']:>+7.4f} "
              f"{r['size_disk_mb']:>7.2f} {flops_str} {r['inference_ms']:>7.2f}")

    print(f"\n  Best: {best['backend']} / {best['observer']}")
    print(f"        Acc={best['accuracy']:.4f}  Drop={best['accuracy_drop']:+.4f}  "
          f"Disk={best['size_disk_mb']:.2f} MB  ×{best.get('compression_ratio', '?'):.2f}")
    print(f"\n  Results saved → {OUTPUT_JSON}")
    print(f"  Models  saved → {MODEL_SAVE_DIR}/")
    print(f"{'='*65}\n")

    return report, all_quantized_models


# %%
if __name__ == "__main__":
    report, quantized_models = main()


  PTQ v2 — ResNet-50 / CIFAR-10
  Math: Q(x)=round(x/S)+Z | S=(x_max-x_min)/(2^b-1) | |ε|≤S/2
  Device: CPU  |  Calibration: 1024 images  |  Latency: single-sample

  Evaluating FP32 baseline ...
  Baseline: Acc=0.9320  Disk=90.05 MB  RAM=89.72 MB  Lat=46.37 ms  GFLOPs=2.623

  [1/3] Dynamic PTQ
        Quantizes  : Linear weights → qint8
        Activations: FP32 (scale computed per batch, on-the-fly)
        Calibration: none required
        → Acc: 0.9319  Drop: +0.0001  Disk: 90.00 MB  Lat: 46.06 ms  GFLOPs: 2.623

  [2/3] Static PTQ — FX-graph mode  (sweeping 4 observers)
        Quantizes : Conv2d + Linear + residual adds (weights + activations → INT8)
        Scale S  = (x_max − x_min) / (2^8 − 1)  from 1024 calib images
        Observer: minmax      

W0428 12:42:02.571000 26632 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


 → Acc: 0.9329  Drop: -0.0009  Disk: 22.99 MB  Lat: 89.55 ms  GFLOPs: 0.000
        Observer: moving_avg   → Acc: 0.9316  Drop: +0.0004  Disk: 22.99 MB  Lat: 187.60 ms  GFLOPs: 0.000
        Observer: histogram    → Acc: 0.9315  Drop: +0.0005  Disk: 22.99 MB  Lat: 188.97 ms  GFLOPs: 0.000
        Observer: percentile   → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  Lat: 188.03 ms  GFLOPs: 0.000

  [3/3] FX-Graph Static PTQ (torch.ao default configs) — x86 + fbgemm
        Uses backend-tuned qconfigs; broader op coverage than observer sweep
        Backend: x86        → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  Lat: 187.84 ms  GFLOPs: 0.000
        Backend: fbgemm     → Acc: 0.9321  Drop: -0.0001  Disk: 22.99 MB  Lat: 187.54 ms  GFLOPs: 0.000

  Saving quantized models to __1__PTQ_quantized_models/ ...
    ✓ resnet50_dynamic.pt                           90.00 MB
    ✓ resnet50_static_minmax.pt                     23.01 MB
    ✓ resnet50_static_moving_avg.pt                 23.01 MB
   